<div style="display: flex; justify-content: space-between; align-items: flex-start; border-bottom: 2px solid #555555; padding-bottom: 15px; margin-bottom: -8px;">
    <div style="width: 50%;">
        <h2><span style="color: #B30033;">▍</span>TRABAJO DE FIN DE MÁSTER</h2>
        <h1 style="margin-top: -10px;">Creación de un framework para recomendar las personas más adecuadas para trabajar en un determinado proyecto de ingeniería del software</h1>
    </div>
    <div style="width: 50%; text-align: right;">
        <div style="display: flex; justify-content: space-between; align-items: flex-start; margin-top: 30px;">
            <div style="width: 20%;"></div>
            <div style="width: 80%; border-left: 2px solid #555555; padding-left: 20px;">
                <div style="margin-bottom: 20px;"><p style="margin: 0; font-size: 1.4em; font-weight: bold;">Profesores</p></div>
                <div style="margin-top: 8px; text-align: right;">
                    <span style="display: block; font-size: 1em; color: #D0D0D0;">Aurora Vizcaíno Barceló</span>
                    <span style="display: block; font-size: 1em; color: #D0D0D0; margin-top: 5px;">Julio Alberto López Gómez</span>
                </div>
            </div>
        </div>
    </div>
</div>

<div style="margin-top: 30px; padding: 20px; background-color: rgba(179, 0, 51, 0.05); border-left: 4px solid #B30033; border-radius: 4px;">
    <h3 style="margin-top: 0; color: #B30033;">🎯 Objetivo de esta Libreta (Fase 3: Minería de Reglas de Asociación — Hard Skills)</h3>
    <p style="font-size: 1.1em; line-height: 1.6; color: #E0E0E0;">
        Partiendo de la matriz transaccional binaria de hard skills (33.880 candidatos, 263 columnas) generada en la Libreta 2, el objetivo es replicar la misma metodología ya validada en soft skills: extracción de reglas con Apriori, FP-Growth y Eclat, filtrado de reglas triviales, y comparación de rendimiento entre algoritmos.
    </p>
    <ul style="font-size: 1.1em; line-height: 1.6; color: #D0D0D0; margin-bottom: 0;">
        <li><strong>Diferencias respecto a soft skills:</strong> el filtrado de circularidad no puede basarse en proxies explícitos (no existen en este pipeline), por lo que se diseñará un criterio basado en la correlación real entre variables numéricas del dataset original.</li>
        <li><strong>Calibración de parámetros:</strong> el <code>min_support</code> se recalibrará específicamente para este dataset, dada su mayor dimensionalidad (263 columnas frente a las 205 de soft skills).</li>
    </ul>
</div>

<div style="border-bottom: 2px solid #555555; padding-bottom: 25px; margin-bottom: 10px">
    <div style="display: flex; align-items: center; margin-bottom: 10px;">
        <span style="color: #B30033; font-size: 1.5em; margin-right: 10px;">▍</span>
        <h3 style="margin: 0; font-size: 1.4em; font-weight: bold">
            Realizado por Miguel Ángel Sánchez-Medina Camacho <span style="font-size: 0.7em; font-style: italic; font-weight: normal;"></span>
        </h3>
    </div>
</div>

### 3.1. Configuración del Entorno y carga de la Matriz Transaccional

Cargamos la matriz transaccional de hard skills generada en la Libreta 2, y la convertimos a tipo booleano, formato requerido por `mlxtend` para un procesamiento eficiente.

In [1]:
import os
import sys
import time
sys.path.append(os.path.abspath(os.path.join('../..')))

from src.hard_skills import config

import pandas as pd
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
from pyECLAT import ECLAT

df_matriz = pd.read_csv(config.MATRIZ_TRANSACCIONAL_PATH, index_col='CandidateId')
df_matriz = df_matriz.fillna(0).astype(bool)

print(f"Dimensiones matriz: {df_matriz.shape}")
display(df_matriz.head(3))

Dimensiones matriz: (33880, 263)


,WorkExp_is_missing,YearsCode_is_missing,ToolCountWork_is_missing,JobSat_is_missing,ConvertedCompYearly_is_missing,EdLevel_is_missing,RemoteWork_is_missing,OrgSize_is_missing,Industry_is_missing,LanguageHaveWorkedWith_JavaScript,...,ToolCountWork_Bin_<=4,ToolCountWork_Bin_>20,ConvertedCompYearly_Bin_100-150k,ConvertedCompYearly_Bin_40-70k,ConvertedCompYearly_Bin_70-100k,ConvertedCompYearly_Bin_<40k,ConvertedCompYearly_Bin_>150k,JobSat_Bin_Alta (8-10),JobSat_Bin_Baja (0-5),JobSat_Bin_Media (6-7)
CandidateId,,,,,,,,,,,,,,,,,,,,,
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,True,False,False,False,True,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,True,False,False,False,False,True,False,False
2,False,False,False,False,False,False,True,True,False,True,...,True,False,False,True,False,False,False,True,False,False


### 3.2. Generación de Itemsets Frecuentes — 1er algoritmo --> Apriori

Propuesto por Agrawal y Srikant en 1994, **Apriori** es uno de los primeros algortimos desarrollados para identificar itemsets frecuentes en bases de datos con transacciones, y posteriormente, conversión a reglas de asociación. Este algoritmo consta de 2 etapas: primero identifica todos los itemsets que ocurren con una frecuencia por encima de un determinado límite y posteriormente a partir de ahí construye combinaciones cada vez más grandes (2 ítems, 3 ítems...), descartando en cada paso cualquier combinación con un subconjunto no frecuente (*downward closure*).

**Conceptos clave utilizados en esta libreta:**
- **Soporte** (`support`): Es el nº de transacciones que contienen el itemset dividido entre el total de transacciones. En nuestro caso, la proporción de candidatos en los que aparece un itemset. El valor mínimo configurado es un min_support de 0.15, el cual significa que la combinación aparece en al menos el 15% de los 88.778 candidatos (~13.300 personas). En un principio el valor iba a ser menor, pero se ha ido ajustando ya que al ejecutar fallaba por problemas de memoria.
- **Confianza** (`confidence`): Podemos definirlo como la probabilidad de que se cumpla el consecuente de una regla, una vez que se cumple el antecedente. Es una probabilidad condicional, no simétrica.
 - **Lift**: Cuánto más frecuente es la combinación de antecedente y consecuente de lo que cabría esperar si fueran independientes. Un lift de 1 indica independencia, pero si ese valor está por encima de 1 tiene asociación positiva.

**Justificación de los parámetros elegidos:**
- `min_support=0.15`: Primero se probaron varios valores, como por ejemplo `min_support=0.05` (el cual es más permisivo), pero con este valor el proceso agotó la memoria disponible del equipo y forzó el cierre de vscode. Esto se produce porque el número de combinaciones candidatas crece de forma exponencial con umbrales bajos. Para evitarlo, se cambió por un umbral más conservador.
- `max_len=4`: Con este parámetro limitamos el tamaño máximo de los itemsets a 4 ítems combinados, lo que ya sería suficiente para una primera exploración interpretable y otra medida adicional de control de memoria.
- `low_memory=True`: Activamos este modo de `mlxtend`, el cual tiene como objetivo que evalúa los itemsets candidatos por lotes en lugar de materializar toda la matriz de combinaciones en memoria al mismo tiempo. También se cambió por el problema de memoria comentado anteriormente.


In [2]:
itemsets_frecuentes = apriori(
    df_matriz,
    min_support=0.15,
    use_colnames=True,
    max_len=4,
    low_memory=True,
    verbose=1
)

pd.set_option('display.max_colwidth', None)
display(itemsets_frecuentes.sort_values('support',ascending=False).head(20))

Processing 6640 combinations | Sampling itemset size 4


,support,itemsets
57,0.788518,frozenset({Employment_Employed})
58,0.711423,frozenset({ICorPM_Individual contributor})
638,0.650502,"frozenset({Employment_Employed, ICorPM_Individual contributor})"
37,0.604132,frozenset({CommPlatformHaveWorkedWith_Stack Overflow})
45,0.571488,frozenset({OfficeStackAsyncHaveWorkedWith_GitHub})
6,0.502007,frozenset({LanguageHaveWorkedWith_JavaScript})
483,0.482025,"frozenset({Employment_Employed, CommPlatformHaveWorkedWith_Stack Overflow})"
38,0.476741,"frozenset({CommPlatformHaveWorkedWith_GitHub (public projects, not private repos)})"
473,0.476476,"frozenset({CommPlatformHaveWorkedWith_Stack Overflow, OfficeStackAsyncHaveWorkedWith_GitHub})"
32,0.474203,frozenset({DevEnvsHaveWorkedWith_Visual Studio Code})


Una vez tenemos el listado con el top 20, vemos que en base al número de soporte, los itemsets individuales dominan la parte alta del ranking, mientras que cualquier combinación de 2 o más ítems solo puede ser tan frecuente como el menos común de sus componentes.

* Destacan `Hobbyist_Yes` (con un 80,3%) Nos dice que la inmensa mayoría de los candidatos programa también como afición, no solo en su empleo. Segundo está `Employment_Employed full-time` (72,5%, señala los empleados que están a tiempo completo) y `WelcomeChange_Just as welcome...` terceri (70,6%, sentido de pertenencia estable a la comunidad)

* Un dato relevante para nuestro proyecto es que `JobFactors_Languages, frameworks, and other technologies I'd be working with` (48,4%) es con diferencia el factor de decisión de empleo más citado entre las respuestas multivalor de `JobFactors` (algo que también veíamos en libretas anteriores). Esto nos confirma que el stack tecnológico sigue siendo el criterio prioritario de los candidatos a la hora de elegir un proyecto, algo muy relevante para el matching final.

### 3.3. Filtrado de reglas: estructura de nulos

Antes de seguir interpretando las reglas obtenidas, vamos a depurarlas y evitar que no queden ninguna que nos afecte las válidas y que no aporten valor alguno. Primero, vamos a ver la **estructura de los valores nulos**.

En la libreta 1 correspondiente al EDA, vimos que habían muchos datos en los proxies de personalidad que tenían mucha correlación con la ausencia de datos en variables laborales como `WorkRemote`, `WorkWeekHrs`, `CodeRevHrs`. En ellas se identifican al mismo subgrupo de candidatos que no están empleados, como pueden ser estudiantes o profesionales desempleados. También pueden contener valores de personas que no quisieron responder, pero para nosotros todos son aquellos que no tienen trabajo.

Sin embargo, aquí hay algunas de esas reglas que deben ser revisadas ya que pueden tener un nivel de confianza muy alto (p.ej  **confianza = 1.000**) que afecten a los resultados de todo el conjunto de reglas. Otro problema a tener en cuenta es que las reglas expertas no contemplan la opción de que exista "ausencia de datos", por lo que sus condiciones se basan en niveles reales de los rasgos (neuroticismo alto/bajo), no en si el dato se conoce o no. Dejar estas reglas mezcladas con el resto sesgaría cualquier comparación posterior no sería buena praxis ya que ocuparían las primeras posiciones por lift sin tener ningún equivalente contra el que contrastarse.

Todo esto lo trataremos más adelante en la libreta

In [3]:
reglas_asociacion = association_rules(itemsets_frecuentes, metric="confidence", min_threshold=0.6)

reglas_asociacion = reglas_asociacion.sort_values('lift', ascending=False)
print(f"Reglas de asociacion obtenidas: {len(reglas_asociacion)}")
display(reglas_asociacion[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(20))

Reglas de asociacion obtenidas: 11257


,antecedents,consequents,support,confidence,lift
3982,frozenset({Continente_Unknown}),"frozenset({ToolCountWork_is_missing, JobSat_is_missing, ConvertedCompYearly_is_missing})",0.167532,0.888124,4.834453
3973,"frozenset({ToolCountWork_is_missing, JobSat_is_missing, ConvertedCompYearly_is_missing})",frozenset({Continente_Unknown}),0.167532,0.911954,4.834453
506,frozenset({Continente_Unknown}),"frozenset({ToolCountWork_is_missing, JobSat_is_missing})",0.167532,0.888124,4.541147
503,"frozenset({ToolCountWork_is_missing, JobSat_is_missing})",frozenset({Continente_Unknown}),0.167532,0.856625,4.541147
3981,"frozenset({Continente_Unknown, ConvertedCompYearly_is_missing})","frozenset({ToolCountWork_is_missing, JobSat_is_missing})",0.167532,0.888124,4.541147
3976,"frozenset({ToolCountWork_is_missing, JobSat_is_missing})","frozenset({Continente_Unknown, ConvertedCompYearly_is_missing})",0.167532,0.856625,4.541147
3977,"frozenset({ToolCountWork_is_missing, Continente_Unknown})","frozenset({JobSat_is_missing, ConvertedCompYearly_is_missing})",0.167532,0.903966,4.264321
3980,"frozenset({JobSat_is_missing, ConvertedCompYearly_is_missing})","frozenset({ToolCountWork_is_missing, Continente_Unknown})",0.167532,0.790309,4.264321
523,"frozenset({JobSat_is_missing, ConvertedCompYearly_is_missing})",frozenset({Continente_Unknown}),0.168123,0.793094,4.204353
525,frozenset({Continente_Unknown}),"frozenset({JobSat_is_missing, ConvertedCompYearly_is_missing})",0.168123,0.891253,4.204353


**Análisis de las reglas obtenidas (sin filtrar):**

Con `min_support=0.15` y `min_confidence=0.6`, Apriori genera **5.324 reglas**. Sin embargo, las 20 con mayor lift están son combinaciones entre `Neuroticismo_Nivel_Desconocido`, `WorkWeekHrs_is_missing`, `CodeRevHrs_is_missing` y `WorkRemote_Bucket_Desconocido (Sin datos)`.

Otras observaciones, como hemos comentado anteriormente son:

1. **Varias reglas alcanzan confianza = 1.000** (ej. `{Neuroticismo_Nivel_Desconocido} → {WorkWeekHrs_is_missing, WorkRemote_Bucket_Desconocido, CodeRevHrs_is_missing}`, fila 2492), por lo que todos los candidatos con neuroticismo desconocido tiene también esas tres variables laborales sin dato.

2. **Muchas filas aparecen en pares con el mismo lift pero distinta confianza** (ej. filas 2503 y 2495, ambas con lift 4.912138). Esto es posible si el lift es simétrico ()`lift(A→B) = lift(B→A)`), porque ambos se calculan como `soporte(A∪B) / (soporte(A) × soporte(B))`. POr el contrario, la confianza no lo es, por eso el mismo itemset genera dos filas con lift idéntico pero confianza distinta según qué lado se tome como antecedente.

3. **Los valores de lift de esta tabla son los más altos de todo el conjunto de reglas** muy por encima de lo que veremos más adelante en las reglas ya tratadas. Esto es porque la presencia de nulos afecta fuertemente a todo el dataset y lo trataremos a lo largo de esta fase.

### 3.3.1. Filtrado de reglas con elementos nulos

Tal cual como hemos comentado antes, vamos a tratar la presencia de valores nulos para evitar que el modelo cometa errores con reglas afectadas. Para ello, haremos el siguiente tratamiento:

Primero identificamos con los sufijos `_is_missing` o `Desconocido` en cualquier ítem de la regla (antecedente o consecuente), y después separamos en un subconjunto de reglas sustantivas: (`reglas_sustantivas`) en vez de borrarlas, así podemos preservar los dos listados:

- El conjunto excluido confirma la consistencia del hallazgo ya reportado en el EDA sobre la estructura no aleatoria del missingness.
- El conjunto `reglas_sustantivas` (5.175 de las 5.324 reglas totales) queda libre de este efecto y es el que se usará en el resto del análisis.

In [4]:
def contiene_item_missing(frozenset_items):
    return any('is_missing' in item or 'Desconocido' in item for item in frozenset_items)

reglas_asociacion['tiene_missing'] = reglas_asociacion.apply(lambda row: contiene_item_missing(row['antecedents']) or contiene_item_missing(row['consequents']),axis=1)
reglas_sustantivas = reglas_asociacion[~reglas_asociacion['tiene_missing']].sort_values('lift', ascending=False)

print(f"Reglas totales: {len(reglas_asociacion)}")
print(f"Reglas con nulos: {reglas_asociacion['tiene_missing'].sum()}")
print(f"Reglas sustantivas restantes: {len(reglas_sustantivas)}")
display(reglas_sustantivas[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(15))

Reglas totales: 11257
Reglas con nulos: 55
Reglas sustantivas restantes: 11202


,antecedents,consequents,support,confidence,lift
495,frozenset({WorkExp_Bin_<5 (Junior)}),frozenset({YearsCode_Bin_<9}),0.164463,0.748221,3.376361
496,frozenset({YearsCode_Bin_<9}),frozenset({WorkExp_Bin_<5 (Junior)}),0.164463,0.742142,3.376361
498,frozenset({WorkExp_Bin_>=19 (Muy Senior)}),frozenset({YearsCode_Bin_>=25}),0.227037,0.885971,3.271575
497,frozenset({YearsCode_Bin_>=25}),frozenset({WorkExp_Bin_>=19 (Muy Senior)}),0.227037,0.838365,3.271575
3967,"frozenset({Employment_Employed, WorkExp_Bin_>=19 (Muy Senior)})",frozenset({YearsCode_Bin_>=25}),0.165702,0.883121,3.261051
3969,frozenset({YearsCode_Bin_>=25}),"frozenset({Employment_Employed, WorkExp_Bin_>=19 (Muy Senior)})",0.165702,0.611880,3.261051
3970,frozenset({WorkExp_Bin_>=19 (Muy Senior)}),"frozenset({Employment_Employed, YearsCode_Bin_>=25})",0.165702,0.646625,3.227886
3966,"frozenset({Employment_Employed, YearsCode_Bin_>=25})",frozenset({WorkExp_Bin_>=19 (Muy Senior)}),0.165702,0.827170,3.227886
8550,"frozenset({PlatformHaveWorkedWith_Docker, OfficeStackAsyncHaveWorkedWith_GitHub, LanguageHaveWorkedWith_Python})",frozenset({PlatformHaveWorkedWith_Pip}),0.158146,0.686747,2.860109
8557,frozenset({PlatformHaveWorkedWith_Pip}),"frozenset({PlatformHaveWorkedWith_Docker, OfficeStackAsyncHaveWorkedWith_GitHub, LanguageHaveWorkedWith_Python})",0.158146,0.658636,2.860109


### 3.3.2. Filtrado de reglas: variables numéricamente redundantes

A diferencia de soft skills, este pipeline no calcula proxies explícitos a partir de otras columnas, por lo que el criterio de circularidad usado allí (comparar cada proxy con sus propios insumos de cálculo) no es aplicable aquí. En su lugar, se identifican pares de variables numéricas del dataset original con una correlación muy alta entre sí —como `WorkExp` y `YearsCode`, con un coeficiente de correlación de Spearman de 0,92 según el análisis de correlaciones de la Libreta 1—, y se excluyen las reglas que combinan sus versiones discretizadas (`WorkExp_Bin`/`YearsCode_Bin`), por describir esencialmente la misma información dos veces en vez de un patrón genuino.

In [5]:
PARES_REDUNDANTES = [
    ('WorkExp_Bin', 'YearsCode_Bin'),
]

def es_regla_redundante(frozenset_items):
    for prefijo_a, prefijo_b in PARES_REDUNDANTES:
        tiene_a = any(item.startswith(prefijo_a) for item in frozenset_items)
        tiene_b = any(item.startswith(prefijo_b) for item in frozenset_items)
        if tiene_a and tiene_b:
            return True
    return False

reglas_sustantivas['es_redundante'] = reglas_sustantivas.apply(
    lambda row: es_regla_redundante(row['antecedents'] | row['consequents']),
    axis=1
)

reglas_limpias = reglas_sustantivas[~reglas_sustantivas['es_redundante']].sort_values('lift', ascending=False)

print(f"Reglas sustantivas previas: {len(reglas_sustantivas)}")
print(f"Reglas redundantes (correlación numérica alta): {reglas_sustantivas['es_redundante'].sum()}")
print(f"Reglas limpias: {len(reglas_limpias)}")
display(reglas_limpias[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(15))

Reglas sustantivas previas: 11202
Reglas redundantes (correlación numérica alta): 14
Reglas limpias: 11188


,antecedents,consequents,support,confidence,lift
8550,"frozenset({PlatformHaveWorkedWith_Docker, OfficeStackAsyncHaveWorkedWith_GitHub, LanguageHaveWorkedWith_Python})",frozenset({PlatformHaveWorkedWith_Pip}),0.158146,0.686747,2.860109
8557,frozenset({PlatformHaveWorkedWith_Pip}),"frozenset({PlatformHaveWorkedWith_Docker, OfficeStackAsyncHaveWorkedWith_GitHub, LanguageHaveWorkedWith_Python})",0.158146,0.658636,2.860109
8549,frozenset({PlatformHaveWorkedWith_Pip}),"frozenset({PlatformHaveWorkedWith_Docker, CommPlatformHaveWorkedWith_Stack Overflow, LanguageHaveWorkedWith_Python})",0.157054,0.654087,2.850222
8542,"frozenset({PlatformHaveWorkedWith_Docker, CommPlatformHaveWorkedWith_Stack Overflow, LanguageHaveWorkedWith_Python})",frozenset({PlatformHaveWorkedWith_Pip}),0.157054,0.684373,2.850222
8564,"frozenset({Employment_Employed, PlatformHaveWorkedWith_Pip})","frozenset({PlatformHaveWorkedWith_Docker, LanguageHaveWorkedWith_Python})",0.150649,0.777930,2.830964
8556,"frozenset({OfficeStackAsyncHaveWorkedWith_GitHub, PlatformHaveWorkedWith_Pip})","frozenset({PlatformHaveWorkedWith_Docker, LanguageHaveWorkedWith_Python})",0.158146,0.777423,2.829119
8547,"frozenset({CommPlatformHaveWorkedWith_Stack Overflow, PlatformHaveWorkedWith_Pip})","frozenset({PlatformHaveWorkedWith_Docker, LanguageHaveWorkedWith_Python})",0.157054,0.771271,2.806731
2067,frozenset({PlatformHaveWorkedWith_Pip}),"frozenset({PlatformHaveWorkedWith_Docker, LanguageHaveWorkedWith_Python})",0.184504,0.768408,2.796312
2064,"frozenset({PlatformHaveWorkedWith_Docker, LanguageHaveWorkedWith_Python})",frozenset({PlatformHaveWorkedWith_Pip}),0.184504,0.671429,2.796312
8565,frozenset({PlatformHaveWorkedWith_Pip}),"frozenset({PlatformHaveWorkedWith_Docker, LanguageHaveWorkedWith_Python, Employment_Employed})",0.150649,0.627412,2.790330


### 3.4. Generación de Itemsets Frecuentes — 2º Algoritmo --> FP-Growth

**FP-Growth** (*Frequent Pattern Growth*) es el segundo algoritmo clásico de minería de reglas de asociación que vamos a emplear en este proyecto. Este algotirmo propuesto en el 2000 permite extraer reglas de asociacion a partir de itemsets frecuentes, pero a diferencia de Apriori, estos se identifican sin necesidad de generar candidatos para cada tamaño, si no que genera y poda combinaciones candidatas de forma iterativa.

En terminos generales, el algoritmo emplea una estructura de árbol (Frequent Pattern tree) donde almacena toda la información de las transacciones. Esta estructura permite comprimir información, haciendo posible que sea más eficiente y dividirlo en datos condicionales. Finalmente, cada partición se analiza de forma separada y se concatenan los resultados obtenidos.

Para los parámetros empleados, vamos a mantener exactamente los mismos que Apriori (`min_support=0.15`, `max_len=4`) para que la comparación entre ambos sea válida. A diferencia de apriori, aquí no disponemos de un parámetro low_memory`, ya que su propia estructura en árbol es, por diseño, más compacta en memoria que la generación de candidatos de Apriori.

In [6]:
start = time.time()
frequent_itemsets_fpgrowth = fpgrowth( df_matriz,min_support=0.15, use_colnames=True,max_len=4)
tiempo_fpgrowth = time.time() - start

print(f"Itemsets frecuentes de FP-Growth: {len(frequent_itemsets_fpgrowth)}")
print(f"Tiempo de ejecución: {tiempo_fpgrowth:.2f} segundos")

Itemsets frecuentes de FP-Growth: 3465
Tiempo de ejecución: 10.96 segundos


**Verificación de equivalencia entre algoritmos**

Ahora que tenemos los dos algoritmos probados vamos a validar que estos, bajo la misma configuración y umbrales sean equivalentes. En caso contrario, destacaríamos aquel que nos ofreciera mejores resultados

In [7]:
reglas_fpgrowth = association_rules(frequent_itemsets_fpgrowth, metric="confidence",min_threshold=0.6)

print(f"Itemsets — Apriori: {len(itemsets_frecuentes)} | FP-Growth: {len(frequent_itemsets_fpgrowth)}")
print(f"Reglas — Apriori: {len(reglas_asociacion)} | FP-Growth: {len(reglas_fpgrowth)}")

Itemsets — Apriori: 3465 | FP-Growth: 3465
Reglas — Apriori: 11257 | FP-Growth: 11257


Si observamos los resultados vemos que Apriori y FP-Growth son algoritmos que nos ofrecen los mismos resultados prácticamente, por lo que cualquiera de los dos nos serían útiles, pero tenemos que comparar cuales de ellos sería más eficiente.



### 3.5. Comparativa de rendimiento: Apriori vs. FP-Growth

Como hemos podido comprobar, tanto Apriori como FP-Growth generan exactamente los mismos itemsets y mismo número de reglas con la misma configuración, por lo que ahora queda la pregunta de cual es más eficiente computacionalmente hablando, en tiempos de ejecución.
Para ello, vamos a volver a ejecutar los dos algortimos con los mismos hiperparámetros y esta vez contamos los tiempos, para ver cual de ellos es más rápido y la diferencia real entre ambas en proporción.



In [8]:
start = time.time()
frequent_itemsets_apriori_tiempos = apriori(df_matriz, min_support=0.15, use_colnames=True, max_len=4, low_memory=True)
tiempo_apriori = time.time() - start

print(f"Tiempo Apriori:   {tiempo_apriori:.2f}s")
print(f"Tiempo FP-Growth: {tiempo_fpgrowth:.2f}s")
print(f"FP-Growth es {tiempo_apriori/tiempo_fpgrowth:.1f}x más rápido" if tiempo_fpgrowth < tiempo_apriori else f"Apriori es {tiempo_fpgrowth/tiempo_apriori:.1f}x más rápido")

Tiempo Apriori:   4.79s
Tiempo FP-Growth: 10.96s
Apriori es 2.3x más rápido


### 3.6. Tercer algoritmo --> Eclat

El tercer algoritmo es **Eclat** (*Equivalence Class Transformation*). Creado en el 2000 por Zaki, su objetivo es encontrar patrones frecuentes. La principal diferencia entre este algoritmo y los anteriores, aquí el dataset no se recorre fila a fila, si no que usa una representación vertical en la que cada item mantiene la lista de identificadores de transacción. Posteriormente, con esa lista de ids realiza un cálculo del soporte mediante la **intersección** de los tidsets de sus componentes. En teoría, esta es una operación muy eficiente, lo que hace de Eclat sea algoritmo atractivo por su complejidad.

Como Eclat no está implementado en `mlxtend`, para este caso recurrimos a `pyECLAT`. Esta librería es la alternativa más extendida para Python, y a diferencia de los dos algoritmos anteriores, `pyECLAT` espera el dataset en formato de **lista de transacciones**, donde cada fila contiene los nombres de los ítems presentes.

**¿Por qué se trabaja con una muestra de 10.000 filas y no con el dataset completo?** Realizando diversas pruebas sobre todas las filas completas con `max_combination=4` (el mismo alcance usado en Apriori y FP-Growth), vemos que la ejecución no se completa: tras más de 50 minutos, la combinación de 3 ítems no termina. POr ello, como alternativa evaulamos el algoritmo sobre una muestra reducida y un alcance menor, lo que es suficiente para poder obtener una medida de rendimiento comparable pudiendo completar el resto de la libreta. Debido a esto, también he cambiado el `max_combination=2`

Una cosa a tener en cuenta es que `pyECLAT` asume que el dataframe que se pasa como entrada tiene un índice contiguo que empieza con 0. Por ello,al tomar una muestra los índices originales que no son contiguos se conserva y provoca un fallo al construir la tabla binaria, por ello hay que incluir un `.reset_index(drop=True)`

Con los resultados que obtengamos aquí vamos a interpretar y comparar con el resto de algoritmos anteriores.

**Transformación a formato de lista de transacciones**

como `pyECLAT` no acepta la matriz binaria `df_matriz` como entrada, es necesario transformarla a una **lista de los nombres de los ítems presentes**, sin los ítems ausentes. Este formato clásico sería por ejemplo: `["Hobbyist_Yes", "Employment_Employed full-time", ...]`.

La función `matriz_a_transacciones` lo que hace es recorrer cada fila de `df_matriz` y, usando el propio booleano de cada columna como máscara (`row.index[row]`), de ellas extrae únicamente los nombres de las columnas activas (`True`) y descarta el resto.

In [9]:
def matriz_a_transacciones(df_bin):
    return df_bin.apply(lambda row: row.index[row].tolist(), axis=1)

transacciones = matriz_a_transacciones(df_matriz)
df_transacciones = pd.DataFrame(transacciones.tolist())

print(f"Forma del dataset de transacciones: {df_transacciones.shape}")
display(df_transacciones.head(3))

Forma del dataset de transacciones: (33880, 180)


,0,1,2,3,4,5,6,7,8,9,...,170,171,172,173,174,175,176,177,178,179
0,LanguageHaveWorkedWith_SQL,LanguageHaveWorkedWith_Bash/Shell (all shells),LanguageHaveWorkedWith_Dart,DatabaseHaveWorkedWith_PostgreSQL,DatabaseHaveWorkedWith_Cloud Firestore,PlatformHaveWorkedWith_Amazon Web Services (AWS),PlatformHaveWorkedWith_Google Cloud,PlatformHaveWorkedWith_Cloudflare,PlatformHaveWorkedWith_Gradle,DevEnvsHaveWorkedWith_Visual Studio Code,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,LanguageHaveWorkedWith_Java,DatabaseHaveWorkedWith_MongoDB,DatabaseHaveWorkedWith_Dynamodb,PlatformHaveWorkedWith_Docker,PlatformHaveWorkedWith_Amazon Web Services (AWS),PlatformHaveWorkedWith_Kubernetes,PlatformHaveWorkedWith_Homebrew,PlatformHaveWorkedWith_Terraform,PlatformHaveWorkedWith_Maven (build tool),WebframeHaveWorkedWith_Spring Boot,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,RemoteWork_is_missing,OrgSize_is_missing,LanguageHaveWorkedWith_JavaScript,LanguageHaveWorkedWith_HTML/CSS,LanguageHaveWorkedWith_TypeScript,LanguageHaveWorkedWith_Dart,DatabaseHaveWorkedWith_PostgreSQL,DatabaseHaveWorkedWith_MySQL,DatabaseHaveWorkedWith_MongoDB,PlatformHaveWorkedWith_npm,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Como cada candidato tiene un número distinto de items activos (por ej. alguien con muchos "Not Applicable" ya excluidos en la Libreta 2 tendrá menos ítems marcados que alguien con el perfil completo) al convertir la lista de listas en un dataframe, pandas iguala todas las filas a la longitud más larga y rellena con `None` los huecos sobrantes. Por lo tanto, el número de columnas (34) corresponde al **máximo de ítems activos que tiene un único candidato** en todo el dataset, a diferencia de las columnas de `df_matriz`.


In [10]:
muestra = df_transacciones.sample(n=10000, random_state=42).reset_index(drop=True)
eclat_instance = ECLAT(data=muestra, verbose=True)

start = time.time()
itemsets_eclat, reglas_eclat_raw = eclat_instance.fit(min_support=0.15, min_combination=1,max_combination=2, separator=' & ')
tiempo_eclat_muestra = time.time() - start

print(f"Tiempo ECLAT para 10k filas, hasta combinaciones de 2: {tiempo_eclat_muestra:.2f}s")
print(f"Itemsets encontrados: {len(itemsets_eclat) if itemsets_eclat is not None else 0}")

100%|██████████| 261/261 [00:00<00:00, 1586.45it/s]


Combination 1 by 1


80it [00:05, 15.83it/s]


Combination 2 by 2


3160it [01:55, 27.44it/s]

Tiempo ECLAT para 10k filas, hasta combinaciones de 2: 120.35s
Itemsets encontrados: 699


### 3.6.1. Resumen comparativo de los tres algoritmos

| Algoritmo | Filas procesadas | Alcance combinación | Tiempo | Itemsets frecuentes |
|---|---|---|---|---|
| Apriori (`low_memory=True`) | 88.778 (completo) | hasta 4 | 10,90s | 2.188 |
| FP-Growth | 88.778 (completo) | hasta 4 | 20,75s | 2.188 |
| Eclat (`pyECLAT`) | 10.000 (muestra) | hasta 2 | 153,23s | 622 |

**Conclusiones del bloque de comparación de algoritmos:**

1. **Apriori y FP-Growth son equivalentes en resultado** (2.188 itemsets, 5.324 reglas idénticas bajo los mismos umbrales), confirmando empíricamente que ambos algoritmos resuelven el mismo problema matemático y solo difieren en la estrategia de recorrido del espacio de búsqueda.

2. **Apriori resultó más rápido que FP-Growth en esta ejecución concreta** (10,90s frente a 20,75s, 1,9x). Esto puede ser por el uso de `low_memory=True` en Apriori (necesario tras agotar la memoria disponible con un `min_support` más bajo en una prueba inicial) combinado con el `min_support=0.15` empleado, que redujo considerablemente el espacio de búsqueda real de Apriori, mientras que FP-Growth incurre en un coste fijo de construcción del FP-Tree sobre todo el dataset que, para este tamaño y umbral, no llegó a compensarse.

3. **Eclat (vía `pyECLAT`) resultó sustancialmente más lento en la práctica**, esto evidencia una brecha entre la complejidad algorítmica teórica de Eclat y la eficiencia real de esta implementación concreta en Python, que recurre a enumeración combinatoria sin vectorización.

En conjunto, estos resultados justifican la elección de **Apriori** como algoritmo principal para el resto del proyecto (dado su buen rendimiento y la posibilidad de operar sobre el dataset completo sin muestreo), reservando FP-Growth como validación de equivalencia y Eclat como referencia comparativa de rendimiento, documentada pero no explotada en profundidad dado su coste computacional en esta implementación.

### 3.7. Exportación de la "Base de Conocimiento"

Las reglas generadas por Apriori (`reglas_asociacion`) y su versión depurada tras el filtrado de nulos y circularidad (`reglas_limpias`) las exportamos ahora a CSV para poder utilizarlas así en fases posteriores del proyecto, donde compararemos más adelante con las reglas expertas de Aurora, para la cual he decidio hacer en una libreta aparte e independiente.

El motivo por que cual exportamos los dos conjuntos es por transparencia, las `reglas_asociacion` (5.324 reglas) como tal es el resultado en bruto y las `reglas_limpias` (4.684 reglas) son aquellas que vamos a comparar en un futuro ya que no tienen problemas con los nulos y problemas circulares por los proxies del Big Five.

En cuanto a la exportación, recordamos que las columnas `antecedents` y `consequents` de `mlxtend` son objetos `frozenset`, los cuales no tienen una representación legible ni reconstruible en un fichero CSV plano. Por ello, antes de exportar aqui convertimos en cadenas de texto con los ítems ordenados alfabéticamente y separados por comas (ej. `frozenset({'A', 'B'})` → `"A, B"`), así el fichero de salida será legible y permitirá realizar comparaciones a futuro.

In [11]:
def exportar_reglas(df_reglas, path):
    df_export = df_reglas.copy()
    df_export['antecedents'] = df_export['antecedents'].apply(lambda x:', '.join(sorted(x)))
    df_export['consequents'] = df_export['consequents'].apply(lambda x:', '.join(sorted(x)))
    df_export.to_csv(path, index=False)
    print(f"Guardado en {path} — {len(df_export)} reglas")

exportar_reglas(reglas_asociacion, config.REGLAS_APRIORI_PATH)
exportar_reglas(reglas_limpias, config.REGLAS_LIMPIAS_PATH)

Guardado en /run/media/miguel-angel/Datos compartidos/TFM/data/processed/hard_skills/reglas_apriori_hard.csv — 11257 reglas
Guardado en /run/media/miguel-angel/Datos compartidos/TFM/data/processed/hard_skills/reglas_limpias_hard.csv — 11188 reglas
